# Этап 3: Предобработка данных

In [1]:
from datasets import load_dataset
from pathlib import Path
import os
from tqdm.notebook import tqdm

In [2]:
RAW_DATA_PATH = Path('../../data/raw/parquet/')
PROCESSED_DATA_PATH = Path('../../data/processed/parquet/')

Path(PROCESSED_DATA_PATH).mkdir(parents=True, exist_ok=True)

In [3]:
def process_batch(batch):
    from bs4 import BeautifulSoup
    import re
    from markdownify import markdownify as md

    USELESS_ALT_TEXTS = {
    'refer to caption',
    '[uncaptioned image]',
    'uncaptioned image',
    '',
    }

    # alt длиннее этого порога считаем осмысленной подписью и сохраняем как [Figure: ...]
    MIN_FIGURE_ALT_LEN = 5

    # Секции-«хвосты», которые выкидываем целиком (по тексту заголовка).
    # Сопоставляем по get_text(), т.к. LaTeXML нумерует заголовки
    # (<span class="ltx_tag">6 </span>References) и .string тогда == None.
    TARGET_SECTIONS = re.compile(r'^(acknowledge?ments?|references|bibliography)$', re.IGNORECASE)
    LEADING_NUMBERING = re.compile(r'^\s*[\dIVXLCivxlc]+(\.[\dIVXLCivxlc]+)*\.?\s*')

    processed_htmls = []

    for html_text in batch['html']:
        # 1. Валидация
        if not html_text or "Conversion to HTML had a Fatal error" in html_text:
            processed_htmls.append(None)
            continue
            
        soup = BeautifulSoup(html_text, 'lxml')
        article = soup.find('article') or soup.find(class_='ltx_page_content')
        
        if not article:
            processed_htmls.append(None)
            continue

        # 2. Очистка мусорных блоков
        garbage_classes = [
            'ltx_bibliography', 'ltx_authors', 'ltx_role_footnotetext', 
            'ltx_navigation', 'ltx_page_footer', 'ltx_is_toc', 
            'mobile-nav', 'header', 'ltx_pagination', 'ltx_ERROR'
        ]
        
        for cls in garbage_classes:
            for tag in article.find_all(class_=cls):
                tag.decompose()
        
        # Удаляем Acknowledgements / References / Bibliography по тексту заголовка
        # (устойчиво к нумерации вида "6 References" и к вложенным span'ам)
        for header in article.find_all(['h1', 'h2', 'h3']):
            heading_text = header.get_text(" ", strip=True)
            cleaned = LEADING_NUMBERING.sub('', heading_text).strip()
            if TARGET_SECTIONS.match(cleaned):
                section = header.find_parent('section')
                if section:
                    section.decompose()

        # 3. Удаление Base64 и технического мусора
        for tag in article.find_all('img', src=re.compile(r'^data:')):
            tag.decompose()
        for tag in article.find_all('a', href=re.compile(r'^data:')):
            tag.decompose()

        # 4. Обработка ссылок
        for a in article.find_all('a'):
            if a.get_text(strip=True):
                a.unwrap()  # .unwrap() удаляет тег, но оставляет содержимое в дереве
            else:
                a.decompose()

        # 5. Обработка картинок
        for img in article.find_all('img'):
            alt = img.get('alt', '')
            if alt.strip().lower() in USELESS_ALT_TEXTS:
                img.decompose()
            elif 'math' in str(img.get('class', [])) or len(alt) > MIN_FIGURE_ALT_LEN:
                img.replace_with(f" [Figure: {alt}] ")
            else:
                img.decompose()

        # 6. Обработка математики (LaTeX): inline -> $...$, display -> $$...$$
        math_registry = {}
        for i, math in enumerate(article.find_all(class_='ltx_Math')):
            latex = math.get('alttext', '')
            if latex:
                placeholder = f"LATEXPH{i}LATEXPH"
                delim = "$$" if math.get('display') == 'block' else "$"
                math_registry[placeholder] = f"{delim}{latex}{delim}"
                math.replace_with(f" {placeholder} ")

        # 7. Конвертация в Markdown (теги a/img уже обработаны выше)
        markdown_text = md(str(article), heading_style="ATX")

        # 8. Возврат математики и финальная зачистка
        for placeholder, original_latex in math_registry.items():
            markdown_text = markdown_text.replace(placeholder, original_latex)
        
        # Убираем лишние переносы строк, которые образуются после удаления блоков
        markdown_text = re.sub(r'\n\s*\n', '\n\n', markdown_text).strip()
            
        processed_htmls.append(markdown_text)

    batch['html'] = processed_htmls
    return batch

In [4]:
ds = load_dataset("parquet", data_files=str(RAW_DATA_PATH / "*.parquet"))

ds_processed = ds.map(
    process_batch,
    batched=True,
    batch_size=50,
    num_proc=os.cpu_count()
)

ds_processed = ds_processed.rename_column('html', 'md')

# Видимость потерь: сколько статей не сконвертировалось (md is None/пусто)
md_col = ds_processed['train']['md']
total = len(md_col)
failed = sum(1 for x in md_col if not x)
converted = total - failed
print(f"Total documents: {total}")
print(f"Converted to MD: {converted} ({converted / total * 100:.1f}%)")
print(f"Failed / empty:  {failed} ({failed / total * 100:.1f}%)")

ds_processed['train'].to_parquet(PROCESSED_DATA_PATH / "processed_data.parquet")

Map (num_proc=12):   0%|          | 0/825 [00:00<?, ? examples/s]

Total documents: 825
Converted to MD: 766 (92.8%)
Failed / empty:  59 (7.2%)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

41873056

In [9]:
import pandas as pd
df = pd.read_parquet('../../data/processed/parquet/processed_data.parquet')

In [13]:
print(df.iloc[0].md)

# Fantastic Reasoning Behaviors and Where to Find Them: Unsupervised Discovery of the Reasoning Process

###### Abstract

Despite the growing reasoning capabilities of recent large language models (LLMs), their internal mechanisms during the reasoning process remain underexplored. Prior approaches often rely on human-defined concepts (e.g., overthinking, reflection) at the word level to analyze reasoning in a supervised manner. However, such methods are limited, as it is infeasible to capture the full spectrum of potential reasoning behaviors, many of which are difficult to define in token space. In this work, we propose an unsupervised framework (namely, RISE: Reasoning behavior Interpretability via Sparse auto-Encoder) for discovering *reasoning vectors*, which we define as directions in the activation space that encode distinct reasoning behaviors. By segmenting chain-of-thought traces into sentence-level ‘steps’ and training sparse auto-encoders (SAEs) on step-level activations, we